In [ ]:
print("[cell] setup: gpu, tesseract, ollama", flush=True)
import subprocess, os, time, json
def sh(cmd, check=True, env=None):
    p = subprocess.Popen(cmd, shell=True, executable="/bin/bash", stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                         env=dict(os.environ, PYTHONUNBUFFERED="1", **(env or {})))
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() and check:
        raise SystemExit(f"command failed (rc {p.returncode}): {cmd}")
    return p.returncode

sh("nvidia-smi --query-gpu=name,memory.total --format=csv || true", check=False)
gpu = subprocess.run("nvidia-smi --query-gpu=name --format=csv,noheader", shell=True, capture_output=True, text=True).stdout.strip()
json.dump({"gpu": gpu}, open("/kaggle/working/times.json", "w"))
sh("apt-get update -qq > /dev/null 2>&1; apt-get install -y -qq zstd tesseract-ocr tesseract-ocr-vie > /dev/null 2>&1 || true", check=False)
sh("tesseract --version 2>&1 | head -1", check=False)
sh("curl -fsSL https://ollama.com/install.sh | sh > /kaggle/working/ollama_install.log 2>&1; tail -2 /kaggle/working/ollama_install.log")
subprocess.Popen("OLLAMA_NUM_PARALLEL=4 ollama serve > /kaggle/working/ollama.log 2>&1", shell=True)
time.sleep(8)
sh("ollama --version")


In [ ]:
print("[cell] unpack GreenScan + pip install", flush=True)
import subprocess, os, time, json
def sh(cmd, check=True, env=None):
    p = subprocess.Popen(cmd, shell=True, executable="/bin/bash", stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                         env=dict(os.environ, PYTHONUNBUFFERED="1", **(env or {})))
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() and check:
        raise SystemExit(f"command failed (rc {p.returncode}): {cmd}")
    return p.returncode

z = subprocess.run("find /kaggle/input -name greenscan_code.zip | head -1", shell=True, capture_output=True, text=True).stdout.strip()
print("bundle:", z, flush=True)
if z:
    sh(f"mkdir -p /kaggle/working/gs && cd /kaggle/working/gs && unzip -q -o '{z}' && ls")
else:  # Kaggle extracted the zip: copy the folder that holds pyproject.toml
    d = subprocess.run("find /kaggle/input -name pyproject.toml | head -1", shell=True, capture_output=True, text=True).stdout.strip()
    print("extracted bundle:", d, flush=True)
    assert d, "neither greenscan_code.zip nor an extracted copy under /kaggle/input"
    sh(f"cp -r '{os.path.dirname(d)}' /kaggle/working/gs && ls /kaggle/working/gs")
sh("cd /kaggle/working/gs && pip install -q -e . 2>&1 | tail -3")
sh("cd /kaggle/working/gs && quantum-agent --help | head -20")


In [ ]:
print("[cell] pull qwen3:4b-instruct-2507-q4_K_M + warm-up", flush=True)
import subprocess, os, time, json
def sh(cmd, check=True, env=None):
    p = subprocess.Popen(cmd, shell=True, executable="/bin/bash", stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                         env=dict(os.environ, PYTHONUNBUFFERED="1", **(env or {})))
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() and check:
        raise SystemExit(f"command failed (rc {p.returncode}): {cmd}")
    return p.returncode

sh("ollama pull qwen3:4b-instruct-2507-q4_K_M 2>&1 | tail -2")
t0 = time.time()
import urllib.request
body = json.dumps({"model": "qwen3:4b-instruct-2507-q4_K_M", "stream": False, "messages": [{"role": "user", "content": "Trả lời một từ: xanh hay đỏ?"}]}).encode()
print(urllib.request.urlopen(urllib.request.Request("http://127.0.0.1:11434/api/chat", body), timeout=600).read()[:400], flush=True)
print("[warm-up]", round(time.time() - t0, 1), "s", flush=True)
sh("ollama ps", check=False)


In [ ]:
print("[cell] HPG rules only (baseline)", flush=True)
import subprocess, os, time, json
def sh(cmd, check=True, env=None):
    p = subprocess.Popen(cmd, shell=True, executable="/bin/bash", stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                         env=dict(os.environ, PYTHONUNBUFFERED="1", **(env or {})))
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() and check:
        raise SystemExit(f"command failed (rc {p.returncode}): {cmd}")
    return p.returncode
E = {'QUANTUM_CONFIG': 'configs/default.yaml', 'QUANTUM_RUNS_DIR': '/kaggle/working/runs', 'QUANTUM_LOG_LEVEL': 'INFO', 'EMBEDDING_PROVIDER': 'lite', 'RERANKER_PROVIDER': 'none', 'QUANTUM_OCR_ENABLED': 'true', 'QUANTUM_OCR_LANGUAGES': 'vie+eng', 'QUANTUM_TESSERACT_COMMAND': 'tesseract', 'QUANTUM_LLM_STANCE': 'off', 'QUANTUM_VERIFY': 'rules', 'QUANTUM_ASSIST_EXPLAIN': 'off', 'QUANTUM_ASSIST_ATTRIBUTES': '0', 'LOCAL_LLM_BASE_URL': ''}
t0 = time.time()
rc = sh("cd /kaggle/working/gs && quantum-agent analyze data/real_cases/sources/originals/HPG_Sustainability_Report_2025.pdf data/real_cases/sources/originals/HPG_Annual_Report_2024.pdf --roles claim_source,evidence --source-types internal,internal 2>&1 | tee /kaggle/working/hpg_rules.log; exit ${PIPESTATUS[0]}", check=False, env=E)
T = json.load(open("/kaggle/working/times.json")); T["hpg_rules"] = round(time.time() - t0, 1); T["hpg_rules_rc"] = rc
json.dump(T, open("/kaggle/working/times.json", "w"))
print("[time] hpg_rules", T["hpg_rules"], "s, rc", rc, flush=True)


In [ ]:
print("[cell] HPG with AI stages (S5 stance, S4 dual reader, S6 explanations)", flush=True)
import subprocess, os, time, json
def sh(cmd, check=True, env=None):
    p = subprocess.Popen(cmd, shell=True, executable="/bin/bash", stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                         env=dict(os.environ, PYTHONUNBUFFERED="1", **(env or {})))
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() and check:
        raise SystemExit(f"command failed (rc {p.returncode}): {cmd}")
    return p.returncode

import threading
stop = threading.Event()
def beat():  # progress while the pipeline runs: how many answers the model has cached so far
    while not stop.wait(60):
        n = subprocess.run("find /kaggle/working/llm_cache -type f 2>/dev/null | wc -l", shell=True, capture_output=True, text=True).stdout.strip()
        print("[progress] llm cache files:", n, flush=True)
threading.Thread(target=beat, daemon=True).start()
E = {'QUANTUM_CONFIG': 'configs/default.yaml', 'QUANTUM_RUNS_DIR': '/kaggle/working/runs', 'QUANTUM_LOG_LEVEL': 'INFO', 'EMBEDDING_PROVIDER': 'lite', 'RERANKER_PROVIDER': 'none', 'QUANTUM_OCR_ENABLED': 'true', 'QUANTUM_OCR_LANGUAGES': 'vie+eng', 'QUANTUM_TESSERACT_COMMAND': 'tesseract', 'QUANTUM_LLM_STANCE': 'on_ambiguous', 'QUANTUM_VERIFY': 'dual', 'QUANTUM_ASSIST_EXPLAIN': 'queue', 'QUANTUM_ASSIST_ATTRIBUTES': '0', 'LLM_PROVIDER': 'local', 'LLM_FALLBACK_ORDER': 'local', 'LOCAL_LLM_PROVIDER': 'ollama', 'LOCAL_LLM_BASE_URL': 'http://127.0.0.1:11434', 'LOCAL_LLM_MODEL': 'qwen3:4b-instruct-2507-q4_K_M', 'LOCAL_LLM_TIMEOUT_SECONDS': '180', 'QUANTUM_LLM_WORKERS': '4', 'QUANTUM_LLM_CACHE_DIR': '/kaggle/working/llm_cache'}
t0 = time.time()
rc = sh("cd /kaggle/working/gs && quantum-agent analyze data/real_cases/sources/originals/HPG_Sustainability_Report_2025.pdf data/real_cases/sources/originals/HPG_Annual_Report_2024.pdf --roles claim_source,evidence --source-types internal,internal 2>&1 | tee /kaggle/working/hpg_ai.log; exit ${PIPESTATUS[0]}", check=False, env=E)
T = json.load(open("/kaggle/working/times.json")); T["hpg_ai"] = round(time.time() - t0, 1); T["hpg_ai_rc"] = rc
json.dump(T, open("/kaggle/working/times.json", "w"))
print("[time] hpg_ai", T["hpg_ai"], "s, rc", rc, flush=True)
stop.set()
sh("tail -5 /kaggle/working/ollama.log", check=False)


In [ ]:
print("[cell] evaluate-real: rules vs AI", flush=True)
import subprocess, os, time, json
def sh(cmd, check=True, env=None):
    p = subprocess.Popen(cmd, shell=True, executable="/bin/bash", stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                         env=dict(os.environ, PYTHONUNBUFFERED="1", **(env or {})))
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() and check:
        raise SystemExit(f"command failed (rc {p.returncode}): {cmd}")
    return p.returncode
E = {'QUANTUM_CONFIG': 'configs/default.yaml', 'QUANTUM_RUNS_DIR': '/kaggle/working/runs', 'QUANTUM_LOG_LEVEL': 'INFO', 'EMBEDDING_PROVIDER': 'lite', 'RERANKER_PROVIDER': 'none', 'QUANTUM_OCR_ENABLED': 'true', 'QUANTUM_OCR_LANGUAGES': 'vie+eng', 'QUANTUM_TESSERACT_COMMAND': 'tesseract', 'QUANTUM_LLM_STANCE': 'off', 'QUANTUM_VERIFY': 'rules', 'QUANTUM_ASSIST_EXPLAIN': 'off', 'QUANTUM_ASSIST_ATTRIBUTES': '0', 'LOCAL_LLM_BASE_URL': ''}
t0 = time.time()
rc = sh("cd /kaggle/working/gs && quantum-agent evaluate-real 2>&1 > /kaggle/working/eval_rules.json | tee /kaggle/working/eval_rules.log; exit ${PIPESTATUS[0]}", check=False, env=E)
T = json.load(open("/kaggle/working/times.json")); T["eval_rules"] = round(time.time() - t0, 1); T["eval_rules_rc"] = rc
json.dump(T, open("/kaggle/working/times.json", "w"))
print("[time] eval_rules", T["eval_rules"], "s, rc", rc, flush=True)
E = {'QUANTUM_CONFIG': 'configs/default.yaml', 'QUANTUM_RUNS_DIR': '/kaggle/working/runs', 'QUANTUM_LOG_LEVEL': 'INFO', 'EMBEDDING_PROVIDER': 'lite', 'RERANKER_PROVIDER': 'none', 'QUANTUM_OCR_ENABLED': 'true', 'QUANTUM_OCR_LANGUAGES': 'vie+eng', 'QUANTUM_TESSERACT_COMMAND': 'tesseract', 'QUANTUM_LLM_STANCE': 'on_ambiguous', 'QUANTUM_VERIFY': 'dual', 'QUANTUM_ASSIST_EXPLAIN': 'queue', 'QUANTUM_ASSIST_ATTRIBUTES': '0', 'LLM_PROVIDER': 'local', 'LLM_FALLBACK_ORDER': 'local', 'LOCAL_LLM_PROVIDER': 'ollama', 'LOCAL_LLM_BASE_URL': 'http://127.0.0.1:11434', 'LOCAL_LLM_MODEL': 'qwen3:4b-instruct-2507-q4_K_M', 'LOCAL_LLM_TIMEOUT_SECONDS': '180', 'QUANTUM_LLM_WORKERS': '4', 'QUANTUM_LLM_CACHE_DIR': '/kaggle/working/llm_cache'}
t0 = time.time()
rc = sh("cd /kaggle/working/gs && quantum-agent evaluate-real 2>&1 > /kaggle/working/eval_ai.json | tee /kaggle/working/eval_ai.log; exit ${PIPESTATUS[0]}", check=False, env=E)
T = json.load(open("/kaggle/working/times.json")); T["eval_ai"] = round(time.time() - t0, 1); T["eval_ai_rc"] = rc
json.dump(T, open("/kaggle/working/times.json", "w"))
print("[time] eval_ai", T["eval_ai"], "s, rc", rc, flush=True)


In [ ]:
print("[cell] summary + pack", flush=True)
import subprocess, os, time, json
def sh(cmd, check=True, env=None):
    p = subprocess.Popen(cmd, shell=True, executable="/bin/bash", stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                         env=dict(os.environ, PYTHONUNBUFFERED="1", **(env or {})))
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() and check:
        raise SystemExit(f"command failed (rc {p.returncode}): {cmd}")
    return p.returncode

import glob, collections
def load(run_id):
    return json.load(open(f"/kaggle/working/runs/{run_id}/result.json", encoding="utf-8"))
def run_id_of(log):
    for line in open(log, encoding="utf-8"):
        if line.startswith("Evidence pack:"):
            return line.split("/runs/")[1].split("/")[0]
def facts(r):
    rel = collections.Counter(e.get("relation_method") or "-" for v in r["verifications"] for e in v["evidence"])
    ir = [x for v in r["verifications"] for x in (v.get("assist") or {}).get("ir", [])]
    expl = [v for v in r["verifications"] if (v.get("assist") or {}).get("explanation")]
    return {"summary": r["summary"], "relation_method": dict(rel), "ir_passages_read": len(ir),
            "ir_vs_rules": dict(collections.Counter(f'{x["rules"]}->{x["ir"]}' for x in ir)),
            "explanations_kept": len(expl), "queue": sum(1 for p in r["priorities"] if p["in_queue"])}
T = json.load(open("/kaggle/working/times.json"))
base, ai = run_id_of("/kaggle/working/hpg_rules.log"), run_id_of("/kaggle/working/hpg_ai.log")
rb, ra = load(base), load(ai)
st = {v["claim"]["claim_id"]: v["status"] for v in rb["verifications"]}
rs = {x["claim_id"]: x["risk_score"] for x in rb["risks"]}
S = {"model": "qwen3:4b-instruct-2507-q4_K_M", "gpu": T.get("gpu"), "runs": {"rules": base, "ai": ai},
     "seconds": {"rules": T["hpg_rules"], "ai": T["hpg_ai"]}, "ratio_ai_over_rules": round(T["hpg_ai"] / T["hpg_rules"], 2),
     "statuses_changed": [(v["claim"]["claim_id"], st.get(v["claim"]["claim_id"]), v["status"]) for v in ra["verifications"]
                          if st.get(v["claim"]["claim_id"]) != v["status"]],
     "risk_scores_changed": sum(1 for x in ra["risks"] if abs(rs.get(x["claim_id"], -1) - x["risk_score"]) > 1e-9),
     "rules": facts(rb), "ai": facts(ra), "llm_cache_files": len(glob.glob("/kaggle/working/llm_cache/**/*", recursive=True)),
     "evaluate_real_seconds": {k: T.get(k) for k in ("eval_rules", "eval_ai")}}
json.dump(S, open("/kaggle/working/summary.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
print(json.dumps({k: S[k] for k in ("runs", "seconds", "ratio_ai_over_rules", "risk_scores_changed")}, ensure_ascii=False), flush=True)
print("statuses changed:", len(S["statuses_changed"]), S["statuses_changed"][:10], flush=True)
print("rules:", json.dumps(S["rules"], ensure_ascii=False), flush=True)
print("ai:   ", json.dumps(S["ai"], ensure_ascii=False), flush=True)

sh("cd /kaggle/working && tar czf runs.tgz runs llm_cache && ls -la /kaggle/working | head -30")
print("ALL CELLS OK", flush=True)
